<a href="https://colab.research.google.com/github/parisazeynaly/RLs-Razor-Reproduction/blob/main/RAZOR_REP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/parisazeynaly/RLs-Razor-Reproduction.git

Cloning into 'RLs-Razor-Reproduction'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 90 (delta 28), reused 4 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 29.46 KiB | 2.27 MiB/s, done.
Resolving deltas: 100% (28/28), done.


In [2]:
%cd RLs-Razor-Reproduction


/content/RLs-Razor-Reproduction


In [3]:
!pip install -r requirements.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00


In [4]:
from huggingface_hub import login
login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
!huggingface-cli whoami


⚠️  Warning: 'huggingface-cli whoami' is deprecated. Use 'hf auth whoami' instead.
parisaze


In [6]:
!pwd
!ls


/content/RLs-Razor-Reproduction
analysis  configs  README.md  requirements.txt	results  src


In [7]:
!pwd
!ls
!find . -name train_sft_llm.py -print


/content/RLs-Razor-Reproduction
analysis  configs  README.md  requirements.txt	results  src
./src/sft/train_sft_llm.py


In [8]:
!sed -n '90,130p' src/sft/train_sft_llm.py


    parser.add_argument("--output_dir", type=str, required=True)
    parser.add_argument("--seed", type=int, default=42)

    parser.add_argument("--max_train_samples", type=int, default=4000)
    parser.add_argument("--max_eval_samples", type=int, default=500)

    parser.add_argument("--max_seq_len", type=int, default=512)
    parser.add_argument("--epochs", type=float, default=1.0)
    parser.add_argument("--lr", type=float, default=2e-4)  # LoRA typically uses higher LR
    parser.add_argument("--per_device_batch_size", type=int, default=1)
    parser.add_argument("--grad_accum", type=int, default=8)
    parser.add_argument("--logging_steps", type=int, default=10)
    parser.add_argument("--save_steps", type=int, default=200)

    # LoRA params (safe defaults)
    parser.add_argument("--lora_r", type=int, default=8)
    parser.add_argument("--lora_alpha", type=int, default=16)
    parser.add_argument("--lora_dropout", type=float, default=0.05)

    args = parser.parse_args()

    s

In [9]:
!grep -n "evaluation_strategy" -n src/sft/train_sft_llm.py


82:        return TrainingArguments(evaluation_strategy="steps", eval_steps=200, **base_kwargs)


In [10]:
!sed -i 's/evaluation_strategy=/eval_strategy=/g' src/sft/train_sft_llm.py


In [11]:
!grep -n "eval_strategy" -n src/sft/train_sft_llm.py


82:        return TrainingArguments(eval_strategy="steps", eval_steps=200, **base_kwargs)
84:        return TrainingArguments(eval_strategy="steps", eval_steps=200, **base_kwargs)


In [12]:
!python src/sft/train_sft_llm.py \
  --base_model Qwen/Qwen2.5-1.5B-Instruct \
  --output_dir results/llm_sft/run_sft_01 \
  --max_train_samples 20 \
  --max_eval_samples 20 \
  --epochs 0.01 \
  --lr 2e-5 \
  --per_device_batch_size 1 \
  --grad_accum 1 \
  --max_seq_len 512


2026-01-04 14:30:39.855537: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767537039.887617    9656 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767537039.897544    9656 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767537039.921853    9656 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767537039.921912    9656 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767537039.921923    9656 computation_placer.cc:177] computation placer alr

In [13]:
%cd /content/RLs-Razor-Reproduction/RLs-Razor-Reproduction
!pwd


[Errno 2] No such file or directory: '/content/RLs-Razor-Reproduction/RLs-Razor-Reproduction'
/content/RLs-Razor-Reproduction
/content/RLs-Razor-Reproduction


In [14]:
!grep -n "SFTTrainer" -n src/sft/train_sft_llm.py || echo "No SFTTrainer found"


No SFTTrainer found


In [15]:
%%writefile src/sft/train_sft_llm.py
import argparse
import os
import json
from typing import Dict, Any

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    set_seed,
)

print("### SFT SCRIPT VERSION: TRANSFORMERS TRAINER ###")
print("RUNNING FILE:", os.path.abspath(__file__))


def build_prompt(question: str, choices: Dict[str, str]) -> str:
    options = "\n".join([f"{k}. {v}" for k, v in choices.items()])
    return (
        "You are a helpful assistant.\n\n"
        "Answer the following multiple-choice question.\n"
        "Choose the correct option and respond with ONLY the letter (A, B, C, or D).\n\n"
        f"Question:\n{question}\n\n"
        f"Options:\n{options}\n\n"
        "Answer:"
    )


def format_sciq_example(ex: Dict[str, Any]) -> Dict[str, str]:
    choices = {
        "A": ex["distractor1"],
        "B": ex["distractor2"],
        "C": ex["distractor3"],
        "D": ex["correct_answer"],  # deterministic: correct is always D
    }
    prompt = build_prompt(ex["question"], choices)
    text = prompt + " D"
    return {"text": text}


def tokenize_fn(tokenizer, max_len: int):
    def _tok(batch):
        out = tokenizer(
            batch["text"],
            truncation=True,
            max_length=max_len,
            padding="max_length",
        )
        out["labels"] = out["input_ids"].copy()
        return out
    return _tok


def make_training_args(output_dir: str, epochs: float, lr: float, bs: int, grad_accum: int, logging_steps: int, save_steps: int):
    base_kwargs = dict(
        output_dir=output_dir,
        num_train_epochs=epochs,
        learning_rate=lr,
        per_device_train_batch_size=bs,
        gradient_accumulation_steps=grad_accum,
        per_device_eval_batch_size=1,
        logging_steps=logging_steps,
        save_steps=save_steps,
        save_total_limit=2,
        bf16=torch.cuda.is_available(),
        fp16=False,
        report_to="none",
        remove_unused_columns=False,
        optim="adamw_torch",
        warmup_steps=50,
        lr_scheduler_type="cosine",
        max_grad_norm=1.0,
    )
    # transformers version compatibility
    try:
        return TrainingArguments(evaluation_strategy="steps", eval_steps=200, **base_kwargs)
    except TypeError:
        return TrainingArguments(eval_strategy="steps", eval_steps=200, **base_kwargs)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--base_model", type=str, required=True)
    parser.add_argument("--output_dir", type=str, required=True)
    parser.add_argument("--seed", type=int, default=42)

    parser.add_argument("--max_train_samples", type=int, default=4000)
    parser.add_argument("--max_eval_samples", type=int, default=500)

    parser.add_argument("--max_seq_len", type=int, default=1024)
    parser.add_argument("--epochs", type=float, default=1.0)
    parser.add_argument("--lr", type=float, default=2e-5)
    parser.add_argument("--per_device_batch_size", type=int, default=1)
    parser.add_argument("--grad_accum", type=int, default=16)
    parser.add_argument("--logging_steps", type=int, default=10)
    parser.add_argument("--save_steps", type=int, default=200)
    args = parser.parse_args()

    set_seed(args.seed)
    os.makedirs(args.output_dir, exist_ok=True)

    with open(os.path.join(args.output_dir, "run_config.json"), "w") as f:
        json.dump(vars(args), f, indent=2)

    print(f"Loading tokenizer/model: {args.base_model}")
    tokenizer = AutoTokenizer.from_pretrained(args.base_model, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        args.base_model,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )
    model.config.use_cache = False

    print("Loading SciQ dataset...")
    ds_train = load_dataset("allenai/sciq", split="train").shuffle(seed=args.seed)
    ds_eval = load_dataset("allenai/sciq", split="validation").shuffle(seed=args.seed)

    ds_train = ds_train.select(range(min(args.max_train_samples, len(ds_train))))
    ds_eval = ds_eval.select(range(min(args.max_eval_samples, len(ds_eval))))

    ds_train = ds_train.map(format_sciq_example, remove_columns=ds_train.column_names)
    ds_eval = ds_eval.map(format_sciq_example, remove_columns=ds_eval.column_names)

    ds_train = ds_train.map(tokenize_fn(tokenizer, args.max_seq_len), batched=True, remove_columns=["text"])
    ds_eval = ds_eval.map(tokenize_fn(tokenizer, args.max_seq_len), batched=True, remove_columns=["text"])

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    training_args = make_training_args(
        output_dir=args.output_dir,
        epochs=args.epochs,
        lr=args.lr,
        bs=args.per_device_batch_size,
        grad_accum=args.grad_accum,
        logging_steps=args.logging_steps,
        save_steps=args.save_steps,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=ds_train,
        eval_dataset=ds_eval,
        data_collator=data_collator,
        tokenizer=tokenizer,
    )

    print("Starting SFT training...")
    trainer.train()

    print("Saving final model...")
    trainer.save_model(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)
    print(f"Done. Saved to: {args.output_dir}")


if __name__ == "__main__":
    main()


Overwriting src/sft/train_sft_llm.py


In [16]:
!grep -n "SFTTrainer" -n src/sft/train_sft_llm.py || echo "OK: no SFTTrainer"


OK: no SFTTrainer


In [17]:
!python src/sft/train_sft_llm.py \
  --base_model Qwen/Qwen2.5-1.5B-Instruct \
  --output_dir results/llm_sft/run_sft_test \
  --max_train_samples 20 \
  --max_eval_samples 20 \
  --epochs 0.01 \
  --lr 2e-5 \
  --per_device_batch_size 1 \
  --grad_accum 1 \
  --max_seq_len 512


2026-01-04 14:31:26.508670: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767537086.528709    9933 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767537086.534792    9933 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767537086.550137    9933 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767537086.550165    9933 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767537086.550169    9933 computation_placer.cc:177] computation placer alr

In [18]:
!pip -q install bitsandbytes


In [19]:
%cd /content/RLs-Razor-Reproduction/RLs-Razor-Reproduction
!pwd


[Errno 2] No such file or directory: '/content/RLs-Razor-Reproduction/RLs-Razor-Reproduction'
/content/RLs-Razor-Reproduction
/content/RLs-Razor-Reproduction


In [20]:
%%writefile src/sft/train_sft_llm.py
import argparse
import os
import json
from typing import Dict, Any

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    set_seed,
    BitsAndBytesConfig,
)

print("### SFT SCRIPT VERSION: 4BIT + CHECKPOINTING (TRANSFORMERS TRAINER) ###")
print("RUNNING FILE:", os.path.abspath(__file__))


def build_prompt(question: str, choices: Dict[str, str]) -> str:
    options = "\n".join([f"{k}. {v}" for k, v in choices.items()])
    return (
        "You are a helpful assistant.\n\n"
        "Answer the following multiple-choice question.\n"
        "Choose the correct option and respond with ONLY the letter (A, B, C, or D).\n\n"
        f"Question:\n{question}\n\n"
        f"Options:\n{options}\n\n"
        "Answer:"
    )


def format_sciq_example(ex: Dict[str, Any]) -> Dict[str, str]:
    # Deterministic labeling: correct is always D
    choices = {
        "A": ex["distractor1"],
        "B": ex["distractor2"],
        "C": ex["distractor3"],
        "D": ex["correct_answer"],
    }
    prompt = build_prompt(ex["question"], choices)
    # Supervised target is the correct letter
    text = prompt + " D"
    return {"text": text}


def tokenize_fn(tokenizer, max_len: int):
    def _tok(batch):
        out = tokenizer(
            batch["text"],
            truncation=True,
            max_length=max_len,
            padding="max_length",
        )
        out["labels"] = out["input_ids"].copy()
        return out
    return _tok


def make_training_args(
    output_dir: str,
    epochs: float,
    lr: float,
    bs: int,
    grad_accum: int,
    logging_steps: int,
    save_steps: int,
):
    base_kwargs = dict(
        output_dir=output_dir,
        num_train_epochs=epochs,
        learning_rate=lr,
        per_device_train_batch_size=bs,
        gradient_accumulation_steps=grad_accum,
        per_device_eval_batch_size=1,
        logging_steps=logging_steps,
        save_steps=save_steps,
        save_total_limit=2,
        bf16=torch.cuda.is_available(),
        fp16=False,
        report_to="none",
        remove_unused_columns=False,
        optim="adamw_torch",
        warmup_steps=50,
        lr_scheduler_type="cosine",
        max_grad_norm=1.0,
    )

    # transformers version compatibility: evaluation_strategy vs eval_strategy
    try:
        return TrainingArguments(evaluation_strategy="steps", eval_steps=200, **base_kwargs)
    except TypeError:
        return TrainingArguments(eval_strategy="steps", eval_steps=200, **base_kwargs)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--base_model", type=str, required=True)
    parser.add_argument("--output_dir", type=str, required=True)
    parser.add_argument("--seed", type=int, default=42)

    parser.add_argument("--max_train_samples", type=int, default=4000)
    parser.add_argument("--max_eval_samples", type=int, default=500)

    parser.add_argument("--max_seq_len", type=int, default=512)  # default safer for Colab
    parser.add_argument("--epochs", type=float, default=1.0)
    parser.add_argument("--lr", type=float, default=2e-5)
    parser.add_argument("--per_device_batch_size", type=int, default=1)
    parser.add_argument("--grad_accum", type=int, default=8)
    parser.add_argument("--logging_steps", type=int, default=10)
    parser.add_argument("--save_steps", type=int, default=200)
    args = parser.parse_args()

    set_seed(args.seed)
    os.makedirs(args.output_dir, exist_ok=True)

    with open(os.path.join(args.output_dir, "run_config.json"), "w") as f:
        json.dump(vars(args), f, indent=2)

    print(f"Loading tokenizer/model (4-bit): {args.base_model}")
    tokenizer = AutoTokenizer.from_pretrained(args.base_model, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 4-bit quantized loading
    compute_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
    )

    model = AutoModelForCausalLM.from_pretrained(
        args.base_model,
        quantization_config=bnb_config,
        device_map="auto",
    )

    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    print("Loading SciQ dataset...")
    ds_train = load_dataset("allenai/sciq", split="train").shuffle(seed=args.seed)
    ds_eval = load_dataset("allenai/sciq", split="validation").shuffle(seed=args.seed)

    ds_train = ds_train.select(range(min(args.max_train_samples, len(ds_train))))
    ds_eval = ds_eval.select(range(min(args.max_eval_samples, len(ds_eval))))

    ds_train = ds_train.map(format_sciq_example, remove_columns=ds_train.column_names)
    ds_eval = ds_eval.map(format_sciq_example, remove_columns=ds_eval.column_names)

    ds_train = ds_train.map(tokenize_fn(tokenizer, args.max_seq_len), batched=True, remove_columns=["text"])
    ds_eval = ds_eval.map(tokenize_fn(tokenizer, args.max_seq_len), batched=True, remove_columns=["text"])

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    training_args = make_training_args(
        output_dir=args.output_dir,
        epochs=args.epochs,
        lr=args.lr,
        bs=args.per_device_batch_size,
        grad_accum=args.grad_accum,
        logging_steps=args.logging_steps,
        save_steps=args.save_steps,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=ds_train,
        eval_dataset=ds_eval,
        data_collator=data_collator,
    )

    print("Starting SFT training...")
    trainer.train()

    print("Saving final model...")
    trainer.save_model(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)
    print(f"Done. Saved to: {args.output_dir}")


if __name__ == "__main__":
    main()


Overwriting src/sft/train_sft_llm.py


In [21]:
!python src/sft/train_sft_llm.py \
  --base_model Qwen/Qwen2.5-1.5B-Instruct \
  --output_dir results/llm_sft/run_sft_test_4bit \
  --max_train_samples 50 \
  --max_eval_samples 50 \
  --epochs 0.05 \
  --lr 2e-5 \
  --per_device_batch_size 1 \
  --grad_accum 4 \
  --max_seq_len 512


2026-01-04 14:32:12.344742: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767537132.377920   10147 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767537132.388150   10147 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767537132.417703   10147 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767537132.417749   10147 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767537132.417758   10147 computation_placer.cc:177] computation placer alr

In [22]:
!pip -q install bitsandbytes peft


In [23]:
%cd /content/RLs-Razor-Reproduction/RLs-Razor-Reproduction


[Errno 2] No such file or directory: '/content/RLs-Razor-Reproduction/RLs-Razor-Reproduction'
/content/RLs-Razor-Reproduction


In [24]:
%%writefile src/sft/train_sft_llm.py
import argparse
import os
import json
from typing import Dict, Any

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    set_seed,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model

print("### SFT SCRIPT VERSION: QLoRA (4bit + LoRA) ###")
print("RUNNING FILE:", os.path.abspath(__file__))


def build_prompt(question: str, choices: Dict[str, str]) -> str:
    options = "\n".join([f"{k}. {v}" for k, v in choices.items()])
    return (
        "You are a helpful assistant.\n\n"
        "Answer the following multiple-choice question.\n"
        "Choose the correct option and respond with ONLY the letter (A, B, C, or D).\n\n"
        f"Question:\n{question}\n\n"
        f"Options:\n{options}\n\n"
        "Answer:"
    )


def format_sciq_example(ex: Dict[str, Any]) -> Dict[str, str]:
    # Deterministic: correct is always D
    choices = {
        "A": ex["distractor1"],
        "B": ex["distractor2"],
        "C": ex["distractor3"],
        "D": ex["correct_answer"],
    }
    prompt = build_prompt(ex["question"], choices)
    text = prompt + " D"
    return {"text": text}


def tokenize_fn(tokenizer, max_len: int):
    def _tok(batch):
        out = tokenizer(
            batch["text"],
            truncation=True,
            max_length=max_len,
            padding="max_length",
        )
        out["labels"] = out["input_ids"].copy()
        return out
    return _tok


def make_training_args(output_dir: str, epochs: float, lr: float, bs: int, grad_accum: int, logging_steps: int, save_steps: int):
    base_kwargs = dict(
        output_dir=output_dir,
        num_train_epochs=epochs,
        learning_rate=lr,
        per_device_train_batch_size=bs,
        gradient_accumulation_steps=grad_accum,
        per_device_eval_batch_size=1,
        logging_steps=logging_steps,
        save_steps=save_steps,
        save_total_limit=2,
        bf16=torch.cuda.is_available(),
        fp16=False,
        report_to="none",
        remove_unused_columns=False,
        optim="adamw_torch",
        warmup_steps=50,
        lr_scheduler_type="cosine",
        max_grad_norm=1.0,
    )
    try:
        return TrainingArguments(evaluation_strategy="steps", eval_steps=200, **base_kwargs)
    except TypeError:
        return TrainingArguments(eval_strategy="steps", eval_steps=200, **base_kwargs)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--base_model", type=str, required=True)
    parser.add_argument("--output_dir", type=str, required=True)
    parser.add_argument("--seed", type=int, default=42)

    parser.add_argument("--max_train_samples", type=int, default=4000)
    parser.add_argument("--max_eval_samples", type=int, default=500)

    parser.add_argument("--max_seq_len", type=int, default=512)
    parser.add_argument("--epochs", type=float, default=1.0)
    parser.add_argument("--lr", type=float, default=2e-4)  # LoRA typically uses higher LR
    parser.add_argument("--per_device_batch_size", type=int, default=1)
    parser.add_argument("--grad_accum", type=int, default=8)
    parser.add_argument("--logging_steps", type=int, default=10)
    parser.add_argument("--save_steps", type=int, default=200)

    # LoRA params (safe defaults)
    parser.add_argument("--lora_r", type=int, default=8)
    parser.add_argument("--lora_alpha", type=int, default=16)
    parser.add_argument("--lora_dropout", type=float, default=0.05)

    args = parser.parse_args()

    set_seed(args.seed)
    os.makedirs(args.output_dir, exist_ok=True)

    with open(os.path.join(args.output_dir, "run_config.json"), "w") as f:
        json.dump(vars(args), f, indent=2)

    print(f"Loading tokenizer/model (4-bit): {args.base_model}")
    tokenizer = AutoTokenizer.from_pretrained(args.base_model, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    compute_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
    )

    model = AutoModelForCausalLM.from_pretrained(
        args.base_model,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    # Attach LoRA adapters (QLoRA)
    # Target modules may vary; these cover many transformer implementations
    lora_config = LoraConfig(
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    print("Loading SciQ dataset...")
    ds_train = load_dataset("allenai/sciq", split="train").shuffle(seed=args.seed)
    ds_eval = load_dataset("allenai/sciq", split="validation").shuffle(seed=args.seed)

    ds_train = ds_train.select(range(min(args.max_train_samples, len(ds_train))))
    ds_eval = ds_eval.select(range(min(args.max_eval_samples, len(ds_eval))))

    ds_train = ds_train.map(format_sciq_example, remove_columns=ds_train.column_names)
    ds_eval = ds_eval.map(format_sciq_example, remove_columns=ds_eval.column_names)

    ds_train = ds_train.map(tokenize_fn(tokenizer, args.max_seq_len), batched=True, remove_columns=["text"])
    ds_eval = ds_eval.map(tokenize_fn(tokenizer, args.max_seq_len), batched=True, remove_columns=["text"])

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    training_args = make_training_args(
        output_dir=args.output_dir,
        epochs=args.epochs,
        lr=args.lr,
        bs=args.per_device_batch_size,
        grad_accum=args.grad_accum,
        logging_steps=args.logging_steps,
        save_steps=args.save_steps,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=ds_train,
        eval_dataset=ds_eval,
        data_collator=data_collator,
    )

    print("Starting SFT training (QLoRA)...")
    trainer.train()

    print("Saving adapters + tokenizer...")
    # With PEFT, this saves adapter weights (small)
    model.save_pretrained(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)
    print(f"Done. Saved to: {args.output_dir}")


if __name__ == "__main__":
    main()


Overwriting src/sft/train_sft_llm.py


In [25]:
!python src/sft/train_sft_llm.py \
  --base_model Qwen/Qwen2.5-1.5B-Instruct \
  --output_dir results/llm_sft/run_sft_test_qlora \
  --max_train_samples 50 \
  --max_eval_samples 50 \
  --epochs 0.05 \
  --lr 2e-4 \
  --per_device_batch_size 1 \
  --grad_accum 4 \
  --max_seq_len 512


2026-01-04 14:33:09.287859: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767537189.327393   10463 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767537189.338046   10463 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767537189.363085   10463 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767537189.363126   10463 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767537189.363135   10463 computation_placer.cc:177] computation placer alr

In [26]:
%cd /content/RLs-Razor-Reproduction/RLs-Razor-Reproduction


[Errno 2] No such file or directory: '/content/RLs-Razor-Reproduction/RLs-Razor-Reproduction'
/content/RLs-Razor-Reproduction


In [27]:
%%writefile src/sft/train_sft_llm.py
import argparse
import os
import json
from typing import Dict, Any

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    set_seed,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("### SFT SCRIPT VERSION: QLoRA (4bit + LoRA) + prepare_model_for_kbit_training ###")
print("RUNNING FILE:", os.path.abspath(__file__))


def build_prompt(question: str, choices: Dict[str, str]) -> str:
    options = "\n".join([f"{k}. {v}" for k, v in choices.items()])
    return (
        "You are a helpful assistant.\n\n"
        "Answer the following multiple-choice question.\n"
        "Choose the correct option and respond with ONLY the letter (A, B, C, or D).\n\n"
        f"Question:\n{question}\n\n"
        f"Options:\n{options}\n\n"
        "Answer:"
    )


def format_sciq_example(ex: Dict[str, Any]) -> Dict[str, str]:
    choices = {
        "A": ex["distractor1"],
        "B": ex["distractor2"],
        "C": ex["distractor3"],
        "D": ex["correct_answer"],
    }
    prompt = build_prompt(ex["question"], choices)
    text = prompt + " D"
    return {"text": text}


def tokenize_fn(tokenizer, max_len: int):
    def _tok(batch):
        out = tokenizer(
            batch["text"],
            truncation=True,
            max_length=max_len,
            padding="max_length",
        )
        out["labels"] = out["input_ids"].copy()
        return out
    return _tok


def make_training_args(output_dir: str, epochs: float, lr: float, bs: int, grad_accum: int, logging_steps: int, save_steps: int):
    base_kwargs = dict(
        output_dir=output_dir,
        num_train_epochs=epochs,
        learning_rate=lr,
        per_device_train_batch_size=bs,
        gradient_accumulation_steps=grad_accum,
        per_device_eval_batch_size=1,
        logging_steps=logging_steps,
        save_steps=save_steps,
        save_total_limit=2,
        bf16=torch.cuda.is_available(),
        fp16=False,
        report_to="none",
        remove_unused_columns=False,
        optim="adamw_torch",
        warmup_steps=50,
        lr_scheduler_type="cosine",
        max_grad_norm=1.0,
    )
    try:
        return TrainingArguments(evaluation_strategy="steps", eval_steps=200, **base_kwargs)
    except TypeError:
        return TrainingArguments(eval_strategy="steps", eval_steps=200, **base_kwargs)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--base_model", type=str, required=True)
    parser.add_argument("--output_dir", type=str, required=True)
    parser.add_argument("--seed", type=int, default=42)

    parser.add_argument("--max_train_samples", type=int, default=4000)
    parser.add_argument("--max_eval_samples", type=int, default=500)

    parser.add_argument("--max_seq_len", type=int, default=512)
    parser.add_argument("--epochs", type=float, default=1.0)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--per_device_batch_size", type=int, default=1)
    parser.add_argument("--grad_accum", type=int, default=8)
    parser.add_argument("--logging_steps", type=int, default=10)
    parser.add_argument("--save_steps", type=int, default=200)

    parser.add_argument("--lora_r", type=int, default=8)
    parser.add_argument("--lora_alpha", type=int, default=16)
    parser.add_argument("--lora_dropout", type=float, default=0.05)

    args = parser.parse_args()

    set_seed(args.seed)
    os.makedirs(args.output_dir, exist_ok=True)

    with open(os.path.join(args.output_dir, "run_config.json"), "w") as f:
        json.dump(vars(args), f, indent=2)

    print(f"Loading tokenizer/model (4-bit): {args.base_model}")
    tokenizer = AutoTokenizer.from_pretrained(args.base_model, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    compute_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
    )

    model = AutoModelForCausalLM.from_pretrained(
        args.base_model,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model.config.use_cache = False

    # IMPORTANT: prepare for k-bit training (enables gradients flow correctly)
    model = prepare_model_for_kbit_training(model)

    # Attach LoRA
    lora_config = LoraConfig(
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    model.train()

    print("Loading SciQ dataset...")
    ds_train = load_dataset("allenai/sciq", split="train").shuffle(seed=args.seed)
    ds_eval = load_dataset("allenai/sciq", split="validation").shuffle(seed=args.seed)

    ds_train = ds_train.select(range(min(args.max_train_samples, len(ds_train))))
    ds_eval = ds_eval.select(range(min(args.max_eval_samples, len(ds_eval))))

    ds_train = ds_train.map(format_sciq_example, remove_columns=ds_train.column_names)
    ds_eval = ds_eval.map(format_sciq_example, remove_columns=ds_eval.column_names)

    ds_train = ds_train.map(tokenize_fn(tokenizer, args.max_seq_len), batched=True, remove_columns=["text"])
    ds_eval = ds_eval.map(tokenize_fn(tokenizer, args.max_seq_len), batched=True, remove_columns=["text"])

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    training_args = make_training_args(
        output_dir=args.output_dir,
        epochs=args.epochs,
        lr=args.lr,
        bs=args.per_device_batch_size,
        grad_accum=args.grad_accum,
        logging_steps=args.logging_steps,
        save_steps=args.save_steps,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=ds_train,
        eval_dataset=ds_eval,
        data_collator=data_collator,
    )

    print("Starting SFT training (QLoRA)...")
    trainer.train()

    print("Saving adapters + tokenizer...")
    model.save_pretrained(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)
    print(f"Done. Saved to: {args.output_dir}")


if __name__ == "__main__":
    main()


Overwriting src/sft/train_sft_llm.py


In [28]:
!python src/sft/train_sft_llm.py \
  --base_model Qwen/Qwen2.5-1.5B-Instruct \
  --output_dir results/llm_sft/run_sft_test_qlora \
  --max_train_samples 50 \
  --max_eval_samples 50 \
  --epochs 0.05 \
  --lr 2e-4 \
  --per_device_batch_size 1 \
  --grad_accum 4 \
  --max_seq_len 512


2026-01-04 14:34:01.524102: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767537241.589857   10721 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767537241.599975   10721 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767537241.624459   10721 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767537241.624498   10721 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767537241.624507   10721 computation_placer.cc:177] computation placer alr

In [29]:
!python src/sft/train_sft_llm.py \
  --base_model Qwen/Qwen2.5-1.5B-Instruct \
  --output_dir results/llm_sft/run_sft_01 \
  --max_train_samples 4000 \
  --max_eval_samples 500 \
  --epochs 1 \
  --lr 2e-4 \
  --per_device_batch_size 1 \
  --grad_accum 8 \
  --max_seq_len 512


2026-01-04 14:34:57.133985: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767537297.172983   11015 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767537297.183234   11015 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767537297.207538   11015 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767537297.207578   11015 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767537297.207587   11015 computation_placer.cc:177] computation placer alr

In [5]:
!pwd
!nvidia-smi


/content/RLs-Razor-Reproduction
/bin/bash: line 1: nvidia-smi: command not found


In [6]:
!head -n 5 src/sft/train_sft_llm.py


import argparse
import os
import json
from dataclasses import dataclass
from typing import Dict, Any, List


In [7]:
%cd /content/RLs-Razor-Reproduction/RLs-Razor-Reproduction
!pwd


[Errno 2] No such file or directory: '/content/RLs-Razor-Reproduction/RLs-Razor-Reproduction'
/content/RLs-Razor-Reproduction
/content/RLs-Razor-Reproduction


In [8]:
%cd /content/RLs-Razor-Reproduction
!pwd


/content/RLs-Razor-Reproduction
/content/RLs-Razor-Reproduction


In [9]:
%cd /content/RLs-Razor-Reproduction
!ls


/content/RLs-Razor-Reproduction
analysis  configs  README.md  requirements.txt	results  src


In [10]:
%%writefile src/sft/train_sft_llm.py
print("### SFT SCRIPT VERSION: QLoRA FINAL (LOCKED) ###")

import argparse
import os
from typing import Dict, Any

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


def build_prompt(question: str, choices: Dict[str, str]) -> str:
    opts = "\n".join([f"{k}. {v}" for k, v in choices.items()])
    return (
        "You are a helpful assistant.\n\n"
        "Answer the following multiple-choice question.\n"
        "Choose the correct option and respond with ONLY the letter (A, B, C, or D).\n\n"
        f"Question:\n{question}\n\nOptions:\n{opts}\n\nAnswer:"
    )


def format_sciq(ex):
    choices = {
        "A": ex["distractor1"],
        "B": ex["distractor2"],
        "C": ex["distractor3"],
        "D": ex["correct_answer"],
    }
    return {"text": build_prompt(ex["question"], choices) + " D"}


def tokenize(tokenizer, max_len):
    def _f(batch):
        out = tokenizer(batch["text"], padding="max_length", truncation=True, max_length=max_len)
        out["labels"] = out["input_ids"].copy()
        return out
    return _f


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base_model", required=True)
    ap.add_argument("--output_dir", required=True)
    ap.add_argument("--max_train_samples", type=int, default=4000)
    ap.add_argument("--epochs", type=float, default=1.0)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--seq_len", type=int, default=512)
    args = ap.parse_args()

    tok = AutoTokenizer.from_pretrained(args.base_model)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        args.base_model,
        quantization_config=bnb,
        device_map="auto",
    )
    model = prepare_model_for_kbit_training(model)

    model = get_peft_model(
        model,
        LoraConfig(
            r=8,
            lora_alpha=16,
            lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            task_type="CAUSAL_LM",
        ),
    )

    ds = load_dataset("allenai/sciq", split="train").shuffle(seed=42)
    ds = ds.select(range(args.max_train_samples))
    ds = ds.map(format_sciq)
    ds = ds.map(tokenize(tok, args.seq_len), batched=True)

    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=args.output_dir,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            num_train_epochs=args.epochs,
            learning_rate=args.lr,
            report_to="none",
        ),
        train_dataset=ds,
        data_collator=DataCollatorForLanguageModeling(tok, mlm=False),
    )

    trainer.train()
    model.save_pretrained(args.output_dir)
    tok.save_pretrained(args.output_dir)


if __name__ == "__main__":
    main()


Overwriting src/sft/train_sft_llm.py


In [11]:
!head -n 5 src/sft/train_sft_llm.py


print("### SFT SCRIPT VERSION: QLoRA FINAL (LOCKED) ###")

import argparse
import os
from typing import Dict, Any


In [14]:
!pip -q install bitsandbytes peft


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 6.8 MB/s eta 0:00:00


In [12]:
!python src/sft/train_sft_llm.py \
  --base_model Qwen/Qwen2.5-1.5B-Instruct \
  --output_dir results/llm_sft/run_sft_01 \
  --max_train_samples 4000 \
  --epochs 1 \
  --lr 2e-4 \
  --seq_len 512


### SFT SCRIPT VERSION: QLoRA FINAL (LOCKED) ###
2026-01-04 22:50:19.296020: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-04 22:50:19.300944: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-04 22:50:19.314550: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767567019.337999    1214 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767567019.344863    1214 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767567019.362480    1214 computation_placer.cc:177] computation placer already 